In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import glob
import tqdm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, KFold, train_test_split

## Functions

In [ ]:
def train_svm_k_folds(df, n_folds):
    X = df.drop("label", axis=1)
    y= df["label"]

    encoder = LabelEncoder()
    y = encoder.fit_transform(y)

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=100)
    kf.get_n_splits(X)

    pipeline_svm = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(random_state=42))
    ])

    param_grid_svm = {
        'svm__C': [0.001, 0.1, 1, 10, 100],        
        'svm__gamma': ['auto'],
        'svm__kernel': ['rbf', 'linear']         
    }

    best_svm_params, best_svm_score = fit_grid_search(pipeline_svm, param_grid_svm, kf, X, y)
    return best_svm_params, best_svm_score
    


In [ ]:
def train_knn_k_folds(df, n_folds):
    X = df.drop("label", axis=1)
    y= df["label"]

    encoder = LabelEncoder()
    y = encoder.fit_transform(y)

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=100)
    kf.get_n_splits(X)

    pipeline_knn = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier())
    ])

    param_grid_knn = {

        "knn__n_neighbors":[1, 3, 5, 9, 13, 17],
        "knn__metric":['euclidean', 'manhattan']         
    }

    best_knn_params, best_knn_score = fit_grid_search(pipeline_knn, param_grid_knn, kf, X, y)
    return best_knn_params, best_knn_score
    
    

In [ ]:
def fit_grid_search(pipeline, param_grid, kf, X, y):
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=kf,
        scoring='accuracy',
        verbose=0,         
        n_jobs=-1 
    )
    grid_search.fit(X, y)

    return grid_search.best_params_, grid_search.best_score_

## Code

In [ ]:
features_dirs = [Path("lbp_features"), Path("glcm_features")]

In [ ]:
results_dict = {}

In [ ]:
for feature_dir in features_dirs:
    print(f"Working on folder: {Path(feature_dir)}")
    for file in tqdm.tqdm(glob.glob(os.path.join(feature_dir, "*.csv"))):
        if Path(feature_dir).name == "lbp_features":
            df = pd.read_csv(file, index_col=0)
            df.drop("filename", axis=1, inplace=True)
            
        else:
            df = pd.read_csv(file)

        if Path(file).name not in results_dict:
            results_dict[Path(file).name] = {}
        
        params, score = train_svm_k_folds(df, 10)
        results_dict[Path(file).name]['svm'] = {'params': params, 'score': score}

        params, score = train_knn_k_folds(df, 10)
        results_dict[Path(file).name]['knn'] = {'params': params, 'score': score}

In [ ]:
records = []
for filename, models in results_dict.items():
    for model_name, results in models.items():
        records.append({
            'feature_set': filename,
            'model': model_name,
            'best_score': results['score'],
            'best_params': results['params']
        })

# Create the final summary DataFrame
summary_df = pd.DataFrame(records)

print("\n--- Final Summary DataFrame ---")
summary_df

In [ ]:
summary_df.to_csv("./sumario_resultados.csv")

In [ ]:
df = pd.read_csv("./lbp_features/lbp_16_2_uniform_3x3.csv",index_col=0)

In [ ]:
X = df.drop(["label", "filename"], axis=1)
y = df["label"]

le = LabelEncoder()
y = le.fit_transform(y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

svm = SVC()

param_grid = {
        'C': [0.001, 0.1, 1, 10, 100],        
        'gamma': ['auto'],
        'kernel': ['rbf', 'linear']         
    }

grid_search = GridSearchCV(
        estimator=svm,
        param_grid=param_grid,
        cv=3,
        scoring='accuracy',
        verbose=0,         
        n_jobs=-1 
    )

grid_search.fit(X_train, y_train)

In [ ]:
best_svm = grid_search.best_estimator_

In [ ]:
y_pred = best_svm.predict(X_test)



In [ ]:
print(classification_report(y_test, y_pred))